# Paper-ball YOLO11n training (Roboflow "paper" dataset, cleaned)

Trains a small YOLO11n paper-ball detector on a Colab GPU and exports it as ONNX for `camera_distance.py`.

**Before you start**
1. `Runtime > Change runtime type > T4 GPU`.
2. Have `dataset_fixed.zip` ready (made by `fix_dataset.py` in `Training/paper_ball_detector/`). The data comes from Roboflow Universe under **CC BY 4.0**: credit the author if you share the model or the data.

Then run the cells from top to bottom. Training takes roughly 10-25 minutes on a T4.

In [ ]:
!nvidia-smi -L || echo "NO GPU: use Runtime > Change runtime type > T4 GPU, then run again"
!pip -q install ultralytics onnx onnxslim

## 1. Upload the dataset zip
Choose `dataset_fixed.zip` (about 80 MB: 248 training, 30 validation, 25 test and 23 real-camera test2 pictures). To use Google Drive instead, see the commented lines.

In [ ]:
import os, zipfile, pathlib
from google.colab import files

# Option B (Drive): put dataset_fixed.zip in My Drive, then uncomment:
# from google.colab import drive; drive.mount('/content/drive')
# !cp /content/drive/MyDrive/dataset_fixed.zip /content/

if not os.path.exists('/content/dataset_fixed.zip'):
    uploaded = files.upload()          # pick dataset_fixed.zip in the dialog

root = pathlib.Path('/content/dataset_fixed')
zipfile.ZipFile('/content/dataset_fixed.zip').extractall(root)

# Write a data.yaml with absolute paths so Ultralytics finds the pictures
import yaml
cfg = {'path': str(root), 'train': 'train/images', 'val': 'valid/images', 'names': {0: 'paper'}}
(root / 'data.yaml').write_text(yaml.safe_dump(cfg))
for split in ('train', 'valid'):
    print(split, len(list((root / split / 'images').glob('*'))), 'pictures')

## 2. Train
Change `EPOCHS` if you like (more epochs = slower, usually a bit better until it stops improving).

In [ ]:
from ultralytics import YOLO

EPOCHS = 80
model = YOLO('yolo11n.pt')             # downloads the small pretrained model
results = model.train(
    data=str(root / 'data.yaml'), epochs=EPOCHS, imgsz=640, batch=16, device=0,
    project='/content/runs', name='roboflow-paperball', exist_ok=True,
    flipud=0.0, plots=False, patience=30,     # stops early if it has not improved for 30 epochs
)
best = '/content/runs/roboflow-paperball/weights/best.pt'
print('best weights:', best)

## 3. Check the result
A high `mAP50` on the validation pictures is expected, because they come from the same photo sessions as the training pictures. The real test is your own camera.

In [ ]:
m = YOLO(best)
metrics = m.val(data=str(root / 'data.yaml'), imgsz=640)
print(f'mAP50 {metrics.box.map50:.3f}   mAP50-95 {metrics.box.map:.3f}')

In [ ]:
import glob, cv2, matplotlib.pyplot as plt
paths = sorted(glob.glob(str(root / 'valid/images/*')))[:8]
preds = m.predict(paths, conf=0.5, imgsz=640, verbose=False)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, r in zip(axes.ravel(), preds):
    ax.imshow(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB)); ax.axis('off')
plt.tight_layout(); plt.show()

## 4. Export and download
This gives `roboflow-paperball.pt` and `roboflow-paperball.onnx`. Put both into `Training/paper_ball_detector/` on your PC.

In [ ]:
import shutil
shutil.copy2(best, '/content/roboflow-paperball.pt')
onnx_path = YOLO('/content/roboflow-paperball.pt').export(format='onnx', imgsz=640, simplify=True, opset=12)
if os.path.abspath(str(onnx_path)) != '/content/roboflow-paperball.onnx':  # the export already lands there
    shutil.copy2(onnx_path, '/content/roboflow-paperball.onnx')
files.download('/content/roboflow-paperball.pt')
files.download('/content/roboflow-paperball.onnx')

## 5. Try it on your camera (on your PC)

```powershell
cd C:\Users\HP\AndroidStudioProjects\LitterVision\Embedded\esp32-cam\scripts
py camera_distance.py --model ..\..\..\Training\paper_ball_detector\roboflow-paperball.onnx --rotate ccw
```

Also score it on the real-camera images with `evaluate_on_camera.py`.

## 4. Test on pictures the model never trained on
Three separate checks, each scored and shown picture by picture (**green box** = the label, **red box** = what the model found, with its confidence):
* **valid**: 30 pictures that were only used to watch training. Optimistic.
* **test1**: 25 held-out pictures of the same kind as the training data. Shows whether the model learned the task.
* **test2**: 23 pictures from the real ESP32-CAM. Shows whether it works on **your camera**.

If test1 is good but test2 is bad, the model does not transfer to your camera: add your own labelled captures to training.

In [ ]:
import yaml, glob, math, cv2, matplotlib.pyplot as plt
from pathlib import Path

SETS = {
    'valid': dict(images=root / 'valid' / 'images', cfg={'path': str(root), 'train': 'train/images', 'val': 'valid/images'}),
    'test1': dict(images=root / 'test' / 'images',  cfg={'path': str(root), 'train': 'train/images', 'val': 'test/images'}),
    'test2': dict(images=root / 'test2' / 'images', cfg={'path': str(root / 'test2'), 'train': 'images', 'val': 'images'}),
}

out = Path('/content/results'); out.mkdir(exist_ok=True)
m = YOLO(best)
header = f"{'set':6} {'pictures':>8} {'precision':>10} {'recall':>8} {'mAP50':>7} {'mAP50-95':>9}"
print(header); lines = [header]
for name, s in SETS.items():
    (root / 'tmp_eval.yaml').write_text(yaml.safe_dump({**s['cfg'], 'names': {0: 'paper'}}))
    r = m.val(data=str(root / 'tmp_eval.yaml'), imgsz=640, plots=False, verbose=False)
    n = len(list(s['images'].glob('*')))
    row = f"{name:6} {n:8d} {r.box.mp:10.3f} {r.box.mr:8.3f} {r.box.map50:7.3f} {r.box.map:9.3f}"
    print(row); lines.append(row)
(out / 'scores.txt').write_text('\n'.join(lines) + '\n')

In [ ]:
def show(name, conf=0.25, cols=6):
    paths = sorted(glob.glob(str(SETS[name]['images'] / '*')))
    preds = m.predict(paths, conf=conf, imgsz=640, verbose=False)
    rows = math.ceil(len(paths) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
    for ax in axes.ravel():
        ax.axis('off')
    for ax, path, r in zip(axes.ravel(), paths, preds):
        img = r.orig_img.copy()
        h, w = img.shape[:2]
        thick = max(2, w // 300)
        label_file = SETS[name]['images'].parent / 'labels' / (Path(path).stem + '.txt')
        if label_file.exists():
            for line in label_file.read_text().splitlines():
                v = line.split()
                if len(v) >= 5:
                    x, y, bw, bh = map(float, v[1:5])
                    cv2.rectangle(img, (int((x - bw / 2) * w), int((y - bh / 2) * h)),
                                  (int((x + bw / 2) * w), int((y + bh / 2) * h)), (0, 200, 0), thick)
        for b in r.boxes:
            x1, y1, x2, y2 = map(int, b.xyxy[0])
            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 0, 255), thick)
            cv2.putText(img, f'{float(b.conf[0]):.2f}', (x1, max(y1 - 6, 20)), cv2.FONT_HERSHEY_SIMPLEX,
                        max(0.7, w / 1400), (0, 0, 255), thick)
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(f'{len(r.boxes)} found', fontsize=8)
    fig.suptitle(f'{name}: {len(paths)} pictures   (green = label, red = prediction)', fontsize=14)
    plt.tight_layout(); fig.savefig(out / f'{name}.png', dpi=70); plt.show()

for name in SETS:
    show(name)

### Download the results
Saves the scores, the three picture sheets and the training numbers as `paperball_results.zip`. Send that file to check the model without copying logs.

In [ ]:
import shutil
from google.colab import files

results_csv = Path('/content/runs/roboflow-paperball/results.csv')   # per-epoch training numbers
if results_csv.exists():
    shutil.copy2(results_csv, out / 'training_results.csv')
shutil.make_archive('/content/paperball_results', 'zip', out)        # scores.txt + valid/test1/test2 pictures + training_results.csv
files.download('/content/paperball_results.zip')